# Competition Model
## ML Modeling Challenge - Wizeline

Entrenamiento y selección de modelos candidatos:
- **Lasso** (baseline lineal)
- **DecisionTree**
- **RandomForest**
- **GradientBoosting**
- **XGBoost**
- **LightGBM**
- **CatBoost**

El mejor modelo se selecciona por el menor **SMAPE de Cross-Validation**.

Se hace selección de variables definitiva vía features importances.

## 0. Imports y configuración

In [ ]:
import yaml
import plotly.express as px
import polars as pl

from src.pipeline.modeling.train_functions import (
    get_feature_importances,
    get_model_pipelines,
    run_cross_validation,
)

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

SEED: int = config["model"]["seed"]
LIST_FEATURE_SELECTED: list[str] = config["features"]["selected_by_mi"]

print(f"Seed global: {SEED}")
print(f"Features cargadas desde config.yaml: {LIST_FEATURE_SELECTED}")


## 1. Carga de datos y selección de features

In [ ]:
df_train = pl.read_csv("data/training_data.csv")
print(f"Shape: {df_train.shape}")
df_train.head()

In [ ]:
X = df_train.select(LIST_FEATURE_SELECTED).to_numpy()
y = df_train["target"].to_numpy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Features seleccionadas: {LIST_FEATURE_SELECTED}")


## 2. Construcción de pipelines

Cada pipeline aplica **MinMaxScaler** seguido del estimador.
Todos los modelos estocásticos usan `random_state=SEED` (5000) para reproducibilidad.

In [ ]:
PIPELINES = get_model_pipelines(seed=SEED)

print(f"Modelos a evaluar: {list(PIPELINES.keys())}")


## 3. Cross-Validation (k=5, shuffle, seed=5000)

Métricas calculadas por fold y promediadas:
- **RMSE** — Root Mean Squared Error
- **MAE** — Mean Absolute Error
- **MAPE** — Mean Absolute Percentage Error
- **SMAPE** — Symmetric MAPE *(métrica de selección)*
- **R²** — Coeficiente de determinación

In [ ]:
cv_results: list[dict] = []

for model_name, pipeline in PIPELINES.items():
    print(f"Evaluando {model_name}...", end=" ")
    metrics = run_cross_validation(pipeline, X, y, n_splits=5, seed=SEED)
    cv_results.append({"model": model_name, **metrics})
    print(
        f"RMSE={metrics['rmse_cv']:.4f} | MAE={metrics['mae_cv']:.4f} "
        f"| SMAPE={metrics['smape_cv']:.4f}% | R²={metrics['r2_cv']:.4f}"
    )

df_cv = pl.DataFrame(cv_results)
print("\nTabla completa de métricas CV:")
print(df_cv)

## 4. Comparación de modelos

In [ ]:
metrics_to_show = ["rmse_cv", "mae_cv", "mape_cv", "smape_cv", "r2_cv"]

fig = px.bar(
    df_cv.sort("smape_cv"),
    x="model",
    y="smape_cv",
    text_auto=".3f",
    color="smape_cv",
    color_continuous_scale="RdYlGn_r",
    title="SMAPE de Cross-Validation por modelo (menor es mejor)",
    labels={"smape_cv": "SMAPE CV (%)", "model": "Modelo"},
)
fig.update_traces(textposition="outside")
fig.update_layout(coloraxis_showscale=False, height=450)
fig.show()

In [ ]:
# Tabla comparativa de todas las métricas
df_cv_display = df_cv.sort("smape_cv")
print("Modelos ordenados por SMAPE CV (ascendente):")
print(df_cv_display)

## 5. Selección del mejor modelo

El **modelo ganador** es el que tiene el menor **SMAPE de CV**.

In [ ]:
best_row = df_cv.sort("smape_cv").row(0, named=True)
best_model_name: str = best_row["model"]
best_smape: float = best_row["smape_cv"]

print(f"Modelo ganador : {best_model_name}")
print(f"SMAPE CV       : {best_smape:.4f}%")
print(f"RMSE CV        : {best_row['rmse_cv']:.4f}")
print(f"MAE CV         : {best_row['mae_cv']:.4f}")
print(f"R² CV          : {best_row['r2_cv']:.4f}")

## 6. Feature importances del modelo ganador

Se entrena el modelo ganador con **todos los datos de entrenamiento** y se grafican las importancias.

In [ ]:
best_pipeline = PIPELINES[best_model_name]
best_pipeline.fit(X, y)

df_importances = get_feature_importances(best_pipeline, LIST_FEATURE_SELECTED)
print(f"Feature importances — {best_model_name}:")
print(df_importances)


In [ ]:
fig = px.bar(
    df_importances,
    x="importance",
    y="feature",
    orientation="h",
    text_auto=".4f",
    color="importance",
    color_continuous_scale="Viridis",
    title=f"Feature Importances — {best_model_name}",
    labels={"importance": "Importancia", "feature": "Feature"},
)
fig.update_traces(textposition="outside")
fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    coloraxis_showscale=False,
    height=500,
)
fig.show()

## 7. Segunda competencia con selección de features

Se filtran del modelo ganador (sección 6) las features con **importancia > 5** y se repite la evaluación completa: Cross-Validation → comparación → mejor modelo.

In [ ]:
IMPORTANCE_THRESHOLD = 5.0

LIST_FEATURE_SELECTED_V2 = (
    df_importances
    .filter(pl.col("importance") > IMPORTANCE_THRESHOLD)
    ["feature"]
    .to_list()
)

print(f"Features con importancia > {IMPORTANCE_THRESHOLD}: {len(LIST_FEATURE_SELECTED_V2)}")
print(LIST_FEATURE_SELECTED_V2)


In [ ]:
X_v2 = df_train.select(LIST_FEATURE_SELECTED_V2).to_numpy()

print(f"X_v2 shape: {X_v2.shape}")


### 7.1 Cross-Validation (k=5)

In [ ]:
pipelines_v2 = get_model_pipelines(seed=SEED)
cv_results_v2: list[dict] = []

for model_name, pipeline in pipelines_v2.items():
    print(f"Evaluando {model_name}...", end=" ")
    metrics = run_cross_validation(pipeline, X_v2, y, n_splits=5, seed=SEED)
    cv_results_v2.append({"model": model_name, **metrics})
    print(
        f"RMSE={metrics['rmse_cv']:.4f} | MAE={metrics['mae_cv']:.4f} "
        f"| SMAPE={metrics['smape_cv']:.4f}% | R²={metrics['r2_cv']:.4f}"
    )

df_cv_v2 = pl.DataFrame(cv_results_v2)
print("\nTabla completa de métricas CV (v2):")
print(df_cv_v2)


### 7.2 Comparación de modelos

In [ ]:
fig = px.bar(
    df_cv_v2.sort("smape_cv"),
    x="model",
    y="smape_cv",
    text_auto=".3f",
    color="smape_cv",
    color_continuous_scale="RdYlGn_r",
    title=f"SMAPE CV por modelo — features con importancia > {IMPORTANCE_THRESHOLD} (menor es mejor)",
    labels={"smape_cv": "SMAPE CV (%)", "model": "Modelo"},
)
fig.update_traces(textposition="outside")
fig.update_layout(coloraxis_showscale=False, height=450)
fig.show()


### 7.3 Mejor modelo (v2)

In [ ]:
best_row_v2 = df_cv_v2.sort("smape_cv").row(0, named=True)
best_model_name_v2: str = best_row_v2["model"]
best_smape_v2: float = best_row_v2["smape_cv"]

print(f"Modelo ganador (v2) : {best_model_name_v2}")
print(f"SMAPE CV            : {best_smape_v2:.4f}%")
print(f"RMSE CV             : {best_row_v2['rmse_cv']:.4f}")
print(f"MAE CV              : {best_row_v2['mae_cv']:.4f}")
print(f"R² CV               : {best_row_v2['r2_cv']:.4f}")

print("\n--- Comparación v1 vs v2 ---")
print(f"SMAPE v1 ({best_model_name:<20}): {best_smape:.4f}%")
print(f"SMAPE v2 ({best_model_name_v2:<20}): {best_smape_v2:.4f}%")
delta = best_smape_v2 - best_smape
print(f"Diferencia          : {delta:+.4f}% ({'mejor' if delta < 0 else 'peor'} con features filtradas)")


In [ ]:
best_pipeline_v2 = pipelines_v2[best_model_name_v2]
best_pipeline_v2.fit(X_v2, y)

df_importances_v2 = get_feature_importances(best_pipeline_v2, LIST_FEATURE_SELECTED_V2)
print(f"Feature importances — {best_model_name_v2} (v2):")
print(df_importances_v2)


In [ ]:
fig = px.bar(
    df_importances_v2,
    x="importance",
    y="feature",
    orientation="h",
    text_auto=".4f",
    color="importance",
    color_continuous_scale="Viridis",
    title=f"Feature Importances — {best_model_name_v2} (v2, importancia > {IMPORTANCE_THRESHOLD})",
    labels={"importance": "Importancia", "feature": "Feature"},
)
fig.update_traces(textposition="outside")
fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    coloraxis_showscale=False,
    height=450,
)
fig.show()
